# CodeAlpha Internship Submission - Task 1: Web Scraping

## Project Title
**Scraping Tutorial Categories and Learning Resources from Tutorials Freak**

## Objective
This notebook is prepared **strictly according to the CodeAlpha Data Analytics Task 1 (Web Scraping)** requirement.

The goal is to:
- use **Python** with **Requests** and **BeautifulSoup**
- extract a **relevant dataset** from a **public website**
- collect the data in **structured tabular form**
- clean the data and export it as a **CSV file**

## Target Website
- **Website:** Tutorials Freak
- **Page used:** Home page

## Dataset Collected
This scraper collects visible learning resources from the page, including:
- section name
- resource title
- resource type
- source page
- scraped timestamp

> This submission focuses only on **Task 1: Web Scraping** and does not add unrelated work from other tasks.


## Step 1 - Import Libraries

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
from pathlib import Path


## Step 2 - Download the Web Page

A request is made to the public webpage.  
For reproducibility, if internet access is not available, the notebook can use a saved HTML snapshot.


In [ ]:
url = "https://www.tutorialsfreak.com/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0 Safari/537.36"
}

snapshot_file = Path("tutorialsfreak_snapshot.html")

try:
    response = requests.get(url, headers=headers, timeout=20)
    response.raise_for_status()
    html_content = response.text
    source_used = "Live website"
except Exception as e:
    if snapshot_file.exists():
        html_content = snapshot_file.read_text(encoding="utf-8")
        source_used = "Local HTML snapshot"
    else:
        raise RuntimeError(f"Unable to fetch live page and snapshot file is missing. Original error: {e}")

print("Source used:", source_used)
print("Total HTML characters:", len(html_content))


## Step 3 - Parse HTML

In [ ]:
soup = BeautifulSoup(html_content, "html.parser")

page_title = soup.title.get_text(strip=True) if soup.title else "No title found"
main_heading = soup.find("h1").get_text(strip=True) if soup.find("h1") else "No H1 found"

print("Page title:", page_title)
print("Main heading:", main_heading)


## Step 4 - Identify Relevant Sections

We will target homepage sections that contain actual learning resources such as:
- interview questions
- quizzes
- programming examples
- live courses


In [ ]:
target_sections = []

for section in soup.find_all("section"):
    heading = section.find("h2")
    items = section.find_all("h3")

    if heading and items:
        target_sections.append({
            "section_name": heading.get_text(" ", strip=True),
            "item_count": len(items)
        })

sections_df = pd.DataFrame(target_sections)
sections_df


## Step 5 - Extract Resource Records

In [ ]:
records = []

for section in soup.find_all("section"):
    heading = section.find("h2")
    if not heading:
        continue

    section_name = heading.get_text(" ", strip=True)
    item_tags = section.find_all("h3")

    for item in item_tags:
        title = item.get_text(" ", strip=True)

        # infer a simple resource type from the section title
        section_lower = section_name.lower()
        if "interview" in section_lower:
            resource_type = "Interview Questions"
        elif "quiz" in section_lower:
            resource_type = "Quiz"
        elif "program" in section_lower:
            resource_type = "Programming Example"
        elif "course" in section_lower or "live" in section_lower:
            resource_type = "Course"
        else:
            resource_type = "Learning Resource"

        records.append({
            "section_name": section_name,
            "resource_title": title,
            "resource_type": resource_type,
            "source_page": url,
            "scraped_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        })

df = pd.DataFrame(records)
df.head()


## Step 6 - Clean the Dataset

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df = df[df["resource_title"].str.len() > 0].reset_index(drop=True)

print("Total records collected:", len(df))
print("Missing values by column:")
print(df.isna().sum())


## Step 7 - Quick Validation

In [ ]:
summary_df = (
    df.groupby(["section_name", "resource_type"])
      .size()
      .reset_index(name="count")
      .sort_values(by=["count", "section_name"], ascending=[False, True])
)

summary_df


## Step 8 - Save the Dataset as CSV

In [ ]:
output_csv = "tutorialsfreak_learning_resources.csv"
df.to_csv(output_csv, index=False)
print(f"Dataset exported successfully: {output_csv}")


## Step 9 - Preview Final Dataset

In [ ]:
df


## Conclusion

This web scraping project successfully:
- accessed a public webpage
- parsed HTML using **BeautifulSoup**
- extracted structured learning-resource data
- organized the data into a **Pandas DataFrame**
- exported the final dataset to **CSV**

This satisfies the core requirement of **CodeAlpha Task 1: Web Scraping** because the work focuses on extracting a relevant dataset from a public website using Python.
